In [1]:
import sys
sys.path.append("../..")

from animgen.core.models.model import BaseModelClass

import numpy as np
from pathlib import Path
import trimesh

In [2]:
SEGMENTED_PATHS = {
    "tail": Path("../../generated_data/test_segmented/tail.glb"),
    "side_fins": Path("../../generated_data/test_segmented/side_fins.glb"),
    "top_fins": Path("../../generated_data/test_segmented/top_fins.glb"),
    "body": Path("../../generated_data/test_segmented/body.glb"),
    "snake_body": Path("../../generated_data/test_segmented/snake_body_stright_test.glb"),
    "snake_complete": Path("../../generated_data/models/paint_mesh_Sea_Snake.glb"),
}

In [3]:
mesh_snake_complete = BaseModelClass(SEGMENTED_PATHS["snake_complete"]).mesh
mesh_snake_body = BaseModelClass(SEGMENTED_PATHS["snake_body"]).mesh

Rendering Multiviews...: 100%|██████████| 20/20 [00:05<00:00,  3.98it/s]


In [4]:
def sample_points_from_mesh(mesh, num_points=1000):
    """
    Sample points from a mesh using uniform sampling.

    Parameters:
    - mesh: The input mesh (trimesh.Trimesh object).
    - num_points: The number of points to sample.

    Returns:
    - sampled_points: A numpy array of shape (num_points, 3) containing the sampled points.
    """
    # Sample points uniformly from the surface of the mesh
    sampled_points, _ = trimesh.sample.sample_surface(mesh, num_points)
    
    return sampled_points

In [8]:
def global_PCA(points):
    """
    Perform PCA on the given points and return the principal components.

    Parameters:
    - points: A numpy array of shape (N, 3) containing the points.

    Returns:
    - pca_components: A numpy array of shape (3, 3) containing the principal components.
    """
    centered_points = points - np.mean(points, axis=0)
    
    cov_matrix = np.cov(centered_points, rowvar=False)
    
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
    
    sorted_indices = np.argsort(eigenvalues)[::-1]
    pca_components = eigenvectors[:, sorted_indices]
    
    return pca_components, eigenvalues[sorted_indices]

In [9]:
def structural_descriptors(eigenvalues):
    """
    Calculate PCA-based structural descriptors from eigenvalues.

    Parameters
    ----------
    eigenvalues : array-like
        Three PCA eigenvalues. They do not need to be pre-sorted.

    Returns
    -------
    dict
        Linearity, planarity, and sphericity.
    """
    eigenvalues = np.asarray(eigenvalues, dtype=float)

    lambda1, lambda2, lambda3 = np.sort(eigenvalues)[::-1]

    if lambda1 <= 0:
        raise ValueError("Largest eigenvalue must be positive.")

    linearity = (lambda1 - lambda2) / lambda1
    planarity = (lambda2 - lambda3) / lambda1
    sphericity = lambda3 / lambda1

    return {
        "linearity": linearity,
        "planarity": planarity,
        "sphericity": sphericity,
    }


In [10]:
print("Snake Straight Body structural descriptors:")
mesh_snake_body_eigenvectors, mesh_snake_body_eigenvalues = global_PCA(mesh_snake_body.vertices)
linearity_mesh_snake_body, planarity_mesh_snake_body, sphericity_mesh_snake_body = \
    structural_descriptors(mesh_snake_body_eigenvalues).values()
print(f"Eigenvalues: {mesh_snake_body_eigenvalues}")
print(f"Linearity: {linearity_mesh_snake_body}")
print(f"Planarity: {planarity_mesh_snake_body}")
print(f"Sphericity: {sphericity_mesh_snake_body}")

print("\nSnake Curved structural descriptors:")
mesh_snake_complete_eigenvectors, mesh_snake_complete_eigenvalues = global_PCA(mesh_snake_complete.vertices)
linearity_mesh_snake_complete, planarity_mesh_snake_complete, sphericity_mesh_snake_complete = \
    structural_descriptors(mesh_snake_complete_eigenvalues).values()
print(f"Eigenvalues: {mesh_snake_complete_eigenvalues}")
print(f"Linearity: {linearity_mesh_snake_complete}")
print(f"Planarity: {planarity_mesh_snake_complete}")
print(f"Sphericity: {sphericity_mesh_snake_complete}")

Snake Straight Body structural descriptors:
Eigenvalues: [0.36425238 0.00082171 0.00082171]
Linearity: 0.9977441053539843
Planarity: 4.54224928196435e-11
Sphericity: 0.0022558946005932943

Snake Curved structural descriptors:
Eigenvalues: [0.2923686  0.06063295 0.00085037]
Linearity: 0.7926147107393443
Planarity: 0.2044767379755225
Sphericity: 0.002908551285133178


In [11]:
""" 
Notes
-----
Global PCA is sensitive to the overall pose of the mesh because it measures the 
global covariance of the vertex distribution. 

For example, consider a snake whose body is progressively bent into an S-shape. 
As the body curves, the global eigenvalues change to reflect the spatial spread 
of the entire mesh rather than its intrinsic tubular morphology. Consequently, 
global PCA may report increased planarity (or even isotropy for tightly coiled 
shapes), despite the snake remaining a long tubular structure. 

This suggests that global PCA is not suitable as the sole descriptor for 
pose-invariant morphology classification. 

Possible improvement 
-------------------- 
Instead of computing PCA on the entire mesh, compute PCA over local neighborhoods 
and aggregate the resulting descriptors. 

For meshes, neighborhood selection based on mesh connectivity (e.g. k-ring neighbors) 
is likely preferable to Euclidean k-nearest neighbors because it: 

- preserves intrinsic surface neighborhoods, 
- avoids connecting nearby but disconnected regions (e.g. coiled snakes), 
- naturally follows the mesh topology. 

This approach still requires experimental validation. 
"""

' \nNotes\n-----\nGlobal PCA is sensitive to the overall pose of the mesh because it measures the \nglobal covariance of the vertex distribution. \n\nFor example, consider a snake whose body is progressively bent into an S-shape. \nAs the body curves, the global eigenvalues change to reflect the spatial spread \nof the entire mesh rather than its intrinsic tubular morphology. Consequently, \nglobal PCA may report increased planarity (or even isotropy for tightly coiled \nshapes), despite the snake remaining a long tubular structure. \n\nThis suggests that global PCA is not suitable as the sole descriptor for \npose-invariant morphology classification. \n\nPossible improvement \n-------------------- \nInstead of computing PCA on the entire mesh, compute PCA over local neighborhoods \nand aggregate the resulting descriptors. \n\nFor meshes, neighborhood selection based on mesh connectivity (e.g. k-ring neighbors) \nis likely preferable to Euclidean k-nearest neighbors because it: \n\n- p